In [1]:
import warnings
warnings.filterwarnings("ignore")

# Data manipulation and analysis
import numpy as np
import pandas as pd

# Planetary Computer tools for STAC API access and authentication
import pystac_client
import planetary_computer as pc
from odc.stac import stac_load
from pystac.extensions.eo import EOExtension as eo

from datetime import date
from tqdm import tqdm
import os
import time
import random

In [2]:
# Setup
tqdm.pandas()

# Reuse a single STAC client (faster + less likely to trigger rate limits)
CATALOG = pystac_client.Client.open(
    "https://planetarycomputer.microsoft.com/api/stac/v1",
    modifier=pc.sign_inplace,
)


def compute_Landsat_values(row, max_retries: int = 5, base_sleep_s: float = 1.0):
    lat = row['Latitude']
    lon = row['Longitude']
    sample_date = pd.to_datetime(row['Sample Date'], dayfirst=True, errors='coerce')

    # Initialize return values with NaN for all bands
    default_return = pd.Series({
        "nir": np.nan,
        "green": np.nan,
        "swir16": np.nan,
        "swir22": np.nan,
        "red": np.nan,
        "blue": np.nan,
        "lwir11": np.nan,
        "coastal": np.nan,
        "qa_pixel": np.nan,
        "qa_aerosol": np.nan,
    })

    if pd.isna(sample_date):
        return default_return

    # Buffer size for ~100m
    bbox_size = 0.00089831
    bbox = [
        lon - bbox_size / 2,
        lat - bbox_size / 2,
        lon + bbox_size / 2,
        lat + bbox_size / 2,
    ]

    # Wider search range, we'll filter to nearest date later
    search = CATALOG.search(
        collections=["landsat-c2-l2"],
        bbox=bbox,
        datetime="2011-01-01/2015-12-31",
        query={"eo:cloud_cover": {"lt": 10}},
    )

    items = search.item_collection()
    if not items:
        return default_return

    # Convert sample date to UTC
    sample_date_utc = sample_date.tz_localize("UTC") if sample_date.tzinfo is None else sample_date.tz_convert("UTC")

    # Band names to attempt (availability depends on sensor)
    bands_of_interest = [
        "green", "nir08", "swir16", "swir22",
        "red", "blue",
        "lwir11",
        "coastal",
        "qa_pixel", "qa_aerosol",
    ]

    last_err = None
    for attempt in range(max_retries):
        try:
            # Pick the item closest to the sample date
            items_sorted = sorted(
                items,
                key=lambda x: abs(pd.to_datetime(x.properties["datetime"]).tz_convert("UTC") - sample_date_utc),
            )
            selected_item = pc.sign(items_sorted[0])

            # Only request assets that exist on this item
            available_bands = [b for b in bands_of_interest if b in selected_item.assets]
            data = stac_load([selected_item], bands=available_bands, bbox=bbox).isel(time=0)

            def safe_median(band_name: str):
                if band_name not in data:
                    return np.nan
                try:
                    band_data = data[band_name].astype("float")
                    median_val = float(band_data.median(skipna=True).values)
                    return median_val if median_val != 0 else np.nan
                except Exception:
                    return np.nan

            return pd.Series({
                "nir": safe_median("nir08"),
                "green": safe_median("green"),
                "swir16": safe_median("swir16"),
                "swir22": safe_median("swir22"),
                "red": safe_median("red"),
                "blue": safe_median("blue"),
                "lwir11": safe_median("lwir11"),
                "coastal": safe_median("coastal"),
                "qa_pixel": safe_median("qa_pixel"),
                "qa_aerosol": safe_median("qa_aerosol"),
            })

        except Exception as e:
            last_err = e
            sleep_s = base_sleep_s * (2 ** attempt) + random.random() * 0.25
            time.sleep(sleep_s)

    # If all retries fail, return NaNs (but don't crash the whole run)
    # Uncomment for debugging:
    # print(f"Failed after {max_retries} retries: {last_err}")
    return default_return


In [6]:
Water_Quality_df=pd.read_csv('../water_quality_training_dataset.csv')
Water_Quality_df.head()

,Latitude,Longitude,Sample Date,Total Alkalinity,Electrical Conductance,Dissolved Reactive Phosphorus
0,-28.760833,17.730278,02-01-2011,128.912,555.0,10.0
1,-26.861111,28.884722,03-01-2011,74.720,162.9,163.0
2,-26.450000,28.085833,03-01-2011,89.254,573.0,80.0
3,-27.671111,27.236944,03-01-2011,82.000,203.6,101.0
4,-27.356667,27.286389,03-01-2011,56.100,145.1,151.0


In [21]:
Water_Quality_df.shape

(9319, 6)

In [22]:
# Use full dataset - all ~9319 rows
# Water_Quality_df_200 = Water_Quality_df.loc[0:199]  # Commented out to process all rows
print(f"Processing all {len(Water_Quality_df)} rows from the training dataset")
Water_Quality_df.shape

Processing all 9319 rows from the training dataset


(9319, 6)

In [ ]:
# Extract band values from Landsat for training dataset (all rows), checkpointing every 200 samples
# This allows you to resume if an API error happens mid-run.

chunk_size = 200
train_features_path = "../New Datasets/landsat_features_training_allbands.csv"

expected_cols = [
    'Latitude', 'Longitude', 'Sample Date',
    'nir', 'green', 'swir16', 'swir22',
    'red', 'blue', 'lwir11', 'coastal', 'qa_pixel', 'qa_aerosol',
    'NDMI', 'MNDWI'
]

def count_rows_in_csv(path: str) -> int:
    # Counts data rows (excluding header). Safe for large files.
    with open(path, 'r', encoding='utf-8') as f:
        return max(sum(1 for _ in f) - 1, 0)


# Determine resume point
start_idx = 0
if os.path.exists(train_features_path):
    existing_header = pd.read_csv(train_features_path, nrows=0).columns.tolist()
    if existing_header != expected_cols:
        raise ValueError(
            f"Existing file has different columns.\n"
            f"Expected: {expected_cols}\n"
            f"Found:    {existing_header}\n"
            f"Fix: delete/rename the existing file or update expected_cols."
        )
    start_idx = count_rows_in_csv(train_features_path)

print("🚀 Running Landsat feature extraction for training data (chunked)...")
print(f"Total rows in training dataset: {len(Water_Quality_df)}")
print(f"Output file: {train_features_path}")
print(f"Chunk size: {chunk_size}")
print(f"Resuming from row index: {start_idx}")

# Process in chunks and append each chunk to CSV
for chunk_start in range(start_idx, len(Water_Quality_df), chunk_size):
    chunk_end = min(chunk_start + chunk_size, len(Water_Quality_df))
    chunk_df = Water_Quality_df.iloc[chunk_start:chunk_end].copy()

    print(f"\nProcessing rows {chunk_start}..{chunk_end-1} ({len(chunk_df)} rows)")

    try:
        chunk_feats = chunk_df.progress_apply(compute_Landsat_values, axis=1)

        # Indices
        eps = 1e-10
        chunk_feats['NDMI'] = (chunk_feats['nir'] - chunk_feats['swir16']) / (chunk_feats['nir'] + chunk_feats['swir16'] + eps)
        chunk_feats['MNDWI'] = (chunk_feats['green'] - chunk_feats['swir16']) / (chunk_feats['green'] + chunk_feats['swir16'] + eps)

        # Join coordinates + date
        chunk_feats['Latitude'] = chunk_df['Latitude'].values
        chunk_feats['Longitude'] = chunk_df['Longitude'].values
        chunk_feats['Sample Date'] = chunk_df['Sample Date'].values

        # Final column order
        chunk_out = chunk_feats[expected_cols]

        # Append to CSV (write header only if file doesn't exist or is empty)
        write_header = (not os.path.exists(train_features_path)) or (count_rows_in_csv(train_features_path) == 0)
        chunk_out.to_csv(train_features_path, mode='a', header=write_header, index=False)

        # Small pause between chunks to be gentle on the API
        time.sleep(0.5)

    except Exception as e:
        print(f"\n❌ Chunk failed at rows {chunk_start}..{chunk_end-1}: {e}")
        print("You can rerun this cell to resume from the last completed chunk.")
        break

# Load whatever has been extracted so far for preview
if os.path.exists(train_features_path):
    landsat_train_features = pd.read_csv(train_features_path)
    print(f"\n✅ Extracted rows so far: {len(landsat_train_features)}")
    display(landsat_train_features.head())
else:
    print("\nNo output file created yet.")

🚀 Running Landsat feature extraction for training data (chunked)...
Total rows in training dataset: 9319
Output file: ../Provided Datasets/landsat_features_training_allbands.csv
Chunk size: 200
Resuming from row index: 0

Processing rows 0..199 (200 rows)


100%|██████████| 200/200 [10:51<00:00,  3.26s/it]



Processing rows 200..399 (200 rows)


100%|██████████| 200/200 [10:18<00:00,  3.09s/it]



Processing rows 400..599 (200 rows)


 40%|███▉      | 79/200 [04:01<06:23,  3.17s/it]

In [ ]:
# (Moved into the chunked extraction cell above)
# NDMI/MNDWI are computed per-chunk before writing to disk.

In [ ]:
# (Moved into the chunked extraction cell above)
# Latitude/Longitude/Sample Date are added per-chunk before writing to disk.

In [ ]:
# (Moved into the chunked extraction cell above)
# Data is appended to CSV after each chunk.

In [ ]:
# Preview File
landsat_train_features.head()

In [ ]:
Validation_df=pd.read_csv('../submission_template.csv')
Validation_df.head()

In [ ]:
Validation_df.shape

In [ ]:
# Extract band values from Landsat for submission dataset (chunked + resumable)
chunk_size = 200
val_features_path = "../New Datasets/landsat_features_validation_allbands.csv"

expected_cols = [
    'Latitude', 'Longitude', 'Sample Date',
    'nir', 'green', 'swir16', 'swir22',
    'red', 'blue', 'lwir11', 'coastal', 'qa_pixel', 'qa_aerosol',
    'NDMI', 'MNDWI'
]

def count_rows_in_csv(path: str) -> int:
    # Counts data rows (excluding header). Safe for large files.
    with open(path, 'r', encoding='utf-8') as f:
        return max(sum(1 for _ in f) - 1, 0)


start_idx = 0
if os.path.exists(val_features_path):
    existing_header = pd.read_csv(val_features_path, nrows=0).columns.tolist()
    if existing_header != expected_cols:
        raise ValueError(
            f"Existing file has different columns.\n"
            f"Expected: {expected_cols}\n"
            f"Found:    {existing_header}\n"
            f"Fix: delete/rename the existing file or update expected_cols."
        )
    start_idx = count_rows_in_csv(val_features_path)

print("🚀 Running Landsat feature extraction for validation data (chunked)...")
print(f"Total rows in validation dataset: {len(Validation_df)}")
print(f"Output file: {val_features_path}")
print(f"Resuming from row index: {start_idx}")

for chunk_start in range(start_idx, len(Validation_df), chunk_size):
    chunk_end = min(chunk_start + chunk_size, len(Validation_df))
    chunk_df = Validation_df.iloc[chunk_start:chunk_end].copy()

    print(f"\nProcessing rows {chunk_start}..{chunk_end-1} ({len(chunk_df)} rows)")

    try:
        chunk_feats = chunk_df.progress_apply(compute_Landsat_values, axis=1)

        eps = 1e-10
        chunk_feats['NDMI'] = (chunk_feats['nir'] - chunk_feats['swir16']) / (chunk_feats['nir'] + chunk_feats['swir16'] + eps)
        chunk_feats['MNDWI'] = (chunk_feats['green'] - chunk_feats['swir16']) / (chunk_feats['green'] + chunk_feats['swir16'] + eps)

        chunk_feats['Latitude'] = chunk_df['Latitude'].values
        chunk_feats['Longitude'] = chunk_df['Longitude'].values
        chunk_feats['Sample Date'] = chunk_df['Sample Date'].values

        chunk_out = chunk_feats[expected_cols]

        write_header = (not os.path.exists(val_features_path)) or (count_rows_in_csv(val_features_path) == 0)
        chunk_out.to_csv(val_features_path, mode='a', header=write_header, index=False)

    except Exception as e:
        print(f"\n❌ Chunk failed at rows {chunk_start}..{chunk_end-1}: {e}")
        print("You can rerun this cell to resume from the last completed chunk.")
        break

if os.path.exists(val_features_path):
    landsat_val_features = pd.read_csv(val_features_path)
    print(f"\n✅ Extracted rows so far: {len(landsat_val_features)}")
    display(landsat_val_features.head())
else:
    print("\nNo output file created yet.")

In [ ]:
# (Moved into the chunked extraction cell above)
# NDMI/MNDWI are computed per-chunk before writing to disk.

In [ ]:
# (Moved into the chunked extraction cell above)
# Latitude/Longitude/Sample Date are added per-chunk before writing to disk.

In [ ]:
# (Moved into the chunked extraction cell above)
# Data is appended to CSV after each chunk.

In [ ]:
# Preview File
landsat_val_features.head()